# 06 — Health Insurance Card: Onboard Members & Verify Coverage

**Time**: ~5 minutes · **Model**: `prebuilt-healthInsuranceCard.us` · **No training required**

---

## Insurance Scenario

When a new member enrolls or visits a healthcare provider, staff need to quickly capture insurance card details:

- Member ID and Group Number
- Plan name and type
- Insurer name
- Copay amounts (specialist, ER, prescription)

This model extracts all of these fields from a photo or scan of a US health insurance card.

---

## Prerequisites

Complete [00-setup-and-basics.ipynb](00-setup-and-basics.ipynb) first.

In [ ]:
import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest, AnalyzeResult

load_dotenv(dotenv_path=os.path.join("..", ".env"))

_api_key = os.environ.get("DOCUMENTINTELLIGENCE_API_KEY", "")
_credential = AzureKeyCredential(_api_key) if _api_key else DefaultAzureCredential()

client = DocumentIntelligenceClient(
    endpoint=os.environ["DOCUMENTINTELLIGENCE_ENDPOINT"],
    credential=_credential
)
print("✅ Client ready")

## Analyze a Health Insurance Card

In [ ]:
# Sample health insurance card
insurance_card_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/rest-api/insurance-card.png"

poller = client.begin_analyze_document(
    "prebuilt-healthInsuranceCard.us",
    AnalyzeDocumentRequest(url_source=insurance_card_url)
)
result: AnalyzeResult = poller.result()

print(f"Insurance cards found: {len(result.documents) if result.documents else 0}")

## Extract Insurance Card Details

In [ ]:
if result.documents:
    card = result.documents[0]
    f = card.fields or {}

    def field_content(field_key):
        field = f.get(field_key)
        if field is None:
            return None, 0
        # SDK v1.0.2 returns DocumentField objects, not dicts.
        content = getattr(field, "content", None)
        confidence = getattr(field, "confidence", 0) or 0
        if not content and isinstance(field, dict):
            content = field.get("content")
            confidence = field.get("confidence", 0)
        return content, confidence

    print("=" * 55)
    print("HEALTH INSURANCE CARD")
    print("=" * 55)

    key_fields = [
        ("Insurer", "Insurance Company"),
        ("GroupNumber", "Group Number"),
        ("IdNumber", "ID Number"),
        ("Member", "Member"),
        ("Plan", "Plan"),
        ("PrescriptionInfo", "Prescription Info"),
    ]

    found_any = False
    for field_key, display_name in key_fields:
        value, conf = field_content(field_key)
        if value:
            found_any = True
            print(f"  {display_name:25s} {str(value):25s} ({conf:.0%})")

    print("\n  COPAY INFORMATION:")
    copay_value, copay_conf = field_content("Copays")
    if copay_value:
        found_any = True
        print(f"    {'Copays':25s} {copay_value} ({copay_conf:.0%})")

    if not found_any:
        print("  No populated fields were returned for this sample card.")
else:
    print("No insurance card detected. Try with a US health insurance card image.")


## Build a Member Enrollment Record

Transform the extracted data into a structure for your member management system.

In [ ]:
import json

if result.documents:
    card = result.documents[0]
    f = card.fields or {}

    def safe_get(field_name):
        field = f.get(field_name)
        if field is None:
            return None
        content = getattr(field, "content", None)
        if content:
            return content
        if isinstance(field, dict):
            return field.get("content")
        return None

    enrollment_record = {
        "insurer": safe_get("Insurer"),
        "group_number": safe_get("GroupNumber"),
        "id_number": safe_get("IdNumber"),
        "member": safe_get("Member"),
        "plan": safe_get("Plan"),
        "copays": safe_get("Copays"),
        "prescription_info": safe_get("PrescriptionInfo"),
        "document_confidence": card.confidence,
    }

    # Keep only keys with data so output is actionable.
    enrollment_record = {k: v for k, v in enrollment_record.items() if v is not None}

    print("Enrollment record:")
    print(json.dumps(enrollment_record, indent=2))


---

## Key Takeaways

| Feature | Value for Insurance |
|---|---|
| **Member ID extraction** | Auto-populate enrollment and claims systems |
| **Group number** | Route to correct plan and benefits configuration |
| **Copay extraction** | Display member cost-sharing at point of service |
| **Coverage verification** | Speed up eligibility checks during provider visits |

## Next Steps

| Notebook | What You'll Learn |
|---|---|
| [07-custom-model.ipynb](07-custom-model.ipynb) | Train a model on your proprietary claim forms |
| [08-document-classifier.ipynb](08-document-classifier.ipynb) | Auto-classify incoming insurance documents |